In [1]:
# Cell 1 — Imports
from dotenv import load_dotenv
import os
load_dotenv(r'C:\Users\Gurveer\ds-portfolio\.env')
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")

import json
import pandas as pd
from openai import OpenAI
from docx import Document
from pptx import Presentation
import openpyxl
import PyPDF2
import markdown
import warnings
warnings.filterwarnings('ignore')

client = OpenAI()
print("All imports successful")

All imports successful


In [2]:
# Cell 2 — Document Reader Engine
def read_pdf(file_path):
    """Extract text from PDF"""
    text = ""
    with open(file_path, 'rb') as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text() + "\n"
    return text.strip()

def read_docx(file_path):
    """Extract text from Word document"""
    doc  = Document(file_path)
    text = "\n".join([para.text for para in doc.paragraphs if para.text.strip()])
    return text

def read_pptx(file_path):
    """Extract text from PowerPoint"""
    prs  = Presentation(file_path)
    text = ""
    for slide_num, slide in enumerate(prs.slides, 1):
        text += f"\n--- Slide {slide_num} ---\n"
        for shape in slide.shapes:
            if hasattr(shape, "text") and shape.text.strip():
                text += shape.text + "\n"
    return text.strip()

def read_xlsx(file_path):
    """Extract data from Excel"""
    wb   = openpyxl.load_workbook(file_path)
    text = ""
    for sheet in wb.sheetnames:
        ws   = wb[sheet]
        text += f"\n--- Sheet: {sheet} ---\n"
        for row in ws.iter_rows(values_only=True):
            row_data = [str(cell) for cell in row if cell is not None]
            if row_data:
                text += " | ".join(row_data) + "\n"
    return text.strip()

def read_txt(file_path):
    """Read plain text file"""
    with open(file_path, 'r', encoding='utf-8', errors='ignore') as f:
        return f.read()

def read_document(file_path):
    """Auto-detect file type and extract text"""
    ext = os.path.splitext(file_path)[1].lower()
    readers = {
        '.pdf':  read_pdf,
        '.docx': read_docx,
        '.pptx': read_pptx,
        '.xlsx': read_xlsx,
        '.txt':  read_txt,
        '.md':   read_txt,
        '.csv':  read_txt
    }
    if ext not in readers:
        return f"Unsupported file type: {ext}"
    return readers[ext](file_path)

# Test with a sample file — create a test document
data_dir = r'C:\Users\Gurveer\ds-portfolio\project-31-document-converter\data'
os.makedirs(data_dir, exist_ok=True)

# Create a sample DOCX to test
test_doc = Document()
test_doc.add_heading('Sample Data Science Report', 0)
test_doc.add_paragraph('This report covers key findings from our analysis.')
test_doc.add_heading('Methodology', 1)
test_doc.add_paragraph('We used XGBoost with SHAP explainability on the Pima Indians dataset.')
test_doc.add_heading('Results', 1)
test_doc.add_paragraph('ROC-AUC: 0.8220. Top features: Glucose, BMI, Age.')
test_doc.add_heading('Conclusion', 1)
test_doc.add_paragraph('The model achieves clinical-grade performance for diabetes risk prediction.')
test_doc_path = f'{data_dir}\\sample_report.docx'
test_doc.save(test_doc_path)

# Test reading
content = read_document(test_doc_path)
print(f"Document read successfully")
print(f"Content length: {len(content)} characters")
print(f"\nExtracted content:")
print(content)

Document read successfully
Content length: 304 characters

Extracted content:
Sample Data Science Report
This report covers key findings from our analysis.
Methodology
We used XGBoost with SHAP explainability on the Pima Indians dataset.
Results
ROC-AUC: 0.8220. Top features: Glucose, BMI, Age.
Conclusion
The model achieves clinical-grade performance for diabetes risk prediction.


In [4]:
# Cell 3 — AI Summarizer & Q&A Engine
def summarize_document(content, style="executive"):
    """
    Summarize document using GPT-4o-mini
    styles: executive, bullet, detailed, technical
    """
    style_prompts = {
        "executive": "Write a 3-sentence executive summary focused on key findings and decisions.",
        "bullet":    "Write a bulleted summary with 5-7 key points, each one sentence.",
        "detailed":  "Write a detailed 2-paragraph summary covering all major sections.",
        "technical": "Write a technical summary focusing on methods, metrics, and results."
    }
    
    prompt = f"""{style_prompts[style]}

Document content:
{content[:3000]}

Return ONLY valid JSON:
{{
  "summary": "your summary here",
  "key_points": ["point1", "point2", "point3"],
  "document_type": "detected document type",
  "word_count": {len(content.split())},
  "topics": ["topic1", "topic2"]
}}"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=800,
        temperature=0.3
    )
    raw = response.choices[0].message.content.strip()
    raw = raw.replace("```json","").replace("```","").strip()
    return json.loads(raw)

def answer_question(content, question):
    """Answer a question about the document"""
    prompt = f"""Answer this question based ONLY on the document content.
If the answer is not in the document, say so clearly.

Document:
{content[:3000]}

Question: {question}

Return ONLY valid JSON:
{{
  "answer": "your answer here",
  "confidence": "high/medium/low",
  "source_quote": "relevant quote from document"
}}"""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=400,
        temperature=0.1
    )
    raw = response.choices[0].message.content.strip()
    raw = raw.replace("```json","").replace("```","").strip()
    return json.loads(raw)

# Test summarization
print("Testing document summarization...")
summary = summarize_document(content, style="technical")

print(f"\n{'='*50}")
print(f"  DOCUMENT SUMMARY")
print(f"{'='*50}")
print(f"Type:       {summary['document_type']}")
print(f"Words:      {summary['word_count']}")
print(f"Topics:     {summary['topics']}")
print(f"\nSummary:\n  {summary['summary']}")
print(f"\nKey Points:")
for p in summary['key_points']:
    print(f"  • {p}")

# Test Q&A
print(f"\n{'='*50}")
print(f"  DOCUMENT Q&A")
print(f"{'='*50}")
questions = [
    "What was the ROC-AUC score?",
    "What dataset was used?",
    "What are the top features?"
]
for q in questions:
    result = answer_question(content, q)
    print(f"\nQ: {q}")
    print(f"A: {result['answer']}")
    print(f"Confidence: {result['confidence']}")

Testing document summarization...

  DOCUMENT SUMMARY
Type:       data science report
Words:      42
Topics:     ['data science', 'machine learning']

Summary:
  The analysis utilized XGBoost with SHAP for explainability on the Pima Indians dataset, achieving a ROC-AUC of 0.8220. Key features identified include Glucose, BMI, and Age, indicating the model's effectiveness in predicting diabetes risk.

Key Points:
  • Method: XGBoost with SHAP
  • ROC-AUC: 0.8220
  • Top features: Glucose, BMI, Age

  DOCUMENT Q&A

Q: What was the ROC-AUC score?
A: 0.8220
Confidence: high

Q: What dataset was used?
A: Pima Indians dataset
Confidence: high

Q: What are the top features?
A: Glucose, BMI, Age
Confidence: high


In [5]:
# Cell 4 — Format Converter
def convert_to_markdown(content, title="Document"):
    """Convert document text to clean Markdown"""
    prompt = f"""Convert this document to clean, well-formatted Markdown.
Add appropriate headers, bullet points, and formatting.

Document content:
{content}

Return only the Markdown text, no JSON wrapper."""

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        max_tokens=1500,
        temperature=0.2
    )
    return response.choices[0].message.content.strip()

def convert_to_docx(content, title="Converted Document"):
    """Convert any text content to formatted DOCX"""
    doc = Document()
    doc.add_heading(title, 0)
    
    lines = content.split('\n')
    for line in lines:
        line = line.strip()
        if not line:
            continue
        if line.startswith('# '):
            doc.add_heading(line[2:], 1)
        elif line.startswith('## '):
            doc.add_heading(line[3:], 2)
        elif line.startswith('- ') or line.startswith('• '):
            p = doc.add_paragraph(style='List Bullet')
            p.add_run(line[2:])
        else:
            doc.add_paragraph(line)
    return doc

def convert_to_html(content, title="Document"):
    """Convert markdown or text to HTML"""
    md_content = f"# {title}\n\n{content}"
    return markdown.markdown(md_content)

# Run conversions
output_dir = r'C:\Users\Gurveer\ds-portfolio\project-31-document-converter\outputs'
os.makedirs(output_dir, exist_ok=True)

print("Converting document to multiple formats...")

# To Markdown
md_content = convert_to_markdown(content, "Sample Data Science Report")
with open(f'{output_dir}\\converted.md', 'w') as f:
    f.write(md_content)
print("✓ Saved as converted.md")

# To DOCX
docx_out = convert_to_docx(md_content, "Sample Data Science Report")
docx_out.save(f'{output_dir}\\converted.docx')
print("✓ Saved as converted.docx")

# To HTML
html_out = convert_to_html(md_content, "Sample Data Science Report")
with open(f'{output_dir}\\converted.html', 'w') as f:
    f.write(html_out)
print("✓ Saved as converted.html")

# Summary JSON
with open(f'{output_dir}\\summary.json', 'w') as f:
    json.dump(summary, f, indent=2)
print("✓ Saved as summary.json")

print(f"\n{'='*50}")
print(f"  CONVERTER SUMMARY")
print(f"{'='*50}")
print(f"  Input formats supported:  PDF, DOCX, PPTX, XLSX, TXT, MD, CSV")
print(f"  Output formats:           Markdown, DOCX, HTML")
print(f"  AI features:              Summarization, Q&A")
print(f"  Exports:                  4 files saved")

Converting document to multiple formats...
✓ Saved as converted.md
✓ Saved as converted.docx
✓ Saved as converted.html
✓ Saved as summary.json

  CONVERTER SUMMARY
  Input formats supported:  PDF, DOCX, PPTX, XLSX, TXT, MD, CSV
  Output formats:           Markdown, DOCX, HTML
  AI features:              Summarization, Q&A
  Exports:                  4 files saved
